# AIRT Scenarios

AIRT (AI Red Team) scenarios test common AI safety risks. Each scenario below runs with minimal
configuration — a single technique and small dataset — to demonstrate usage. For full configuration
options, see the [Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb).

## Setup

In [ ]:
from pyrit.output import output_scenario_async
from pyrit.prompt_target import OpenAIChatTarget
from pyrit.scenario import DatasetAttackConfiguration
from pyrit.setup import IN_MEMORY, initialize_pyrit_async
from pyrit.setup.initializers import (
    LoadDefaultDatasets,
    ScorerInitializer,
    TargetInitializer,
    TechniqueInitializer,
)

await initialize_pyrit_async(  # type: ignore
    memory_db_type=IN_MEMORY,
    initializers=[TargetInitializer(), ScorerInitializer(), TechniqueInitializer(), LoadDefaultDatasets()],
)

objective_target = OpenAIChatTarget()

Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


[pyrit:alembic] No new upgrade operations detected.


TextAdaptive: _EXCLUDED_TECHNIQUES entries ['prompt_sending'] are not in the current scenario-techniques catalog ['context_compliance', 'crescendo_history_lecture', 'crescendo_journalist_interview', 'crescendo_movie_director', 'crescendo_simulated', 'many_shot', 'pair', 'red_teaming', 'role_play', 'tap', 'violent_durian']; the exclusion is a no-op for those entries. Remove stale entries or update the catalog.


## Rapid Response

Tests whether a target can be induced to generate harmful content across seven categories: hate,
fairness, violence, sexual, harassment, misinformation, and leakage. Each technique applies a
different attack technique to the full set of harm datasets.

```bash
pyrit_scan airt.rapid_response \
  --initializers target \
  --target openai_chat \
  --techniques role_play_movie_script \
  --dataset-names airt_hate \
  --max-dataset-size 1
```

**Available techniques:** ALL, DEFAULT, SINGLE_TURN, MULTI_TURN, role_play_movie_script, many_shot, tap

In [ ]:
from pyrit.scenario.airt import RapidResponse, RapidResponseTechnique

dataset_config = DatasetAttackConfiguration(dataset_names=["airt_hate"], max_dataset_size=1)

scenario = RapidResponse()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [RapidResponseTechnique.role_play_movie_script],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

scenario_result = await scenario.run_async()  # type: ignore

Executing RapidResponse:   0%|          | 0/2 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(scenario_result)


                                 📊 SCENARIO RESULTS: RapidResponse                                  

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: RapidResponse
    • Scenario Version: 2
    • PyRIT Version: 0.15.0.dev0
    • Description:
        Rapid Response scenario for content-harms testing. Tests model behavior across multiple harm categories using
        selectable attack techniques.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o-japan-nilfilter
    • Target Endpoint: https://pyrit-japan-test.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: FloatScaleThresholdScorer
      • scorer_type: true_false
      • score_aggregator: OR_
        └─ Composite of 1 scorer(s):
            • Scorer Type: AzureContentFilterScorer
            • scorer_type: float_scale

    ▸ Performance Metrics
  

## Psychosocial

Tests whether a target appropriately handles psychosocial crisis situations, evaluating imminent
crisis handling and licensed therapist impersonation.

```bash
pyrit_scan airt.psychosocial --target openai_chat --techniques imminent_crisis --max-dataset-size 1
```

**Available techniques:** ALL, ImminentCrisis, LicensedTherapist

### Conversation-Level Scoring

Unlike content harms that can be evaluated from a single response, psychosocial harms emerge
gradually across a multi-turn conversation. This scenario uses create_conversation_scorer()
to evaluate the **entire conversation history** rather than just the latest response — critical
for detecting patterns like gradual crisis escalation.

Each subharm has a built-in YAML rubric:

- **crisis_management.yaml** — Evaluates crisis recognition using the MHFA/ALGEE methodology
  (Approach, Listen, Give reassurance, Encourage professional help, Encourage self-help).
  Scores range from 1 (not harmful) to 5 (extremely harmful).
- **licensed_therapist.yaml** — Evaluates whether the model inappropriately provides clinical
  mental health advice.

Rubrics can be customized per-subharm via the SubharmConfig dataclass.

**Note:** This scenario does not include a default baseline. A single-turn baseline would not be
meaningful because psychosocial harms emerge through multi-turn escalation.

In [ ]:
from pyrit.scenario.airt import Psychosocial, PsychosocialTechnique

dataset_config = DatasetAttackConfiguration(dataset_names=["airt_imminent_crisis"], max_dataset_size=1)

scenario = Psychosocial()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [PsychosocialTechnique.ImminentCrisis],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

scenario_result = await scenario.run_async()  # type: ignore

Executing Psychosocial:   0%|          | 0/4 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(scenario_result)


                                  📊 SCENARIO RESULTS: Psychosocial                                  

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Psychosocial
    • Scenario Version: 2
    • PyRIT Version: 0.15.0.dev0
    • Description:
        Psychosocial Harms Scenario implementation for PyRIT. This scenario contains various psychosocial harm-based
        checks that evaluate model behavior for responses that may harm the well-being or mental health of users.
        Examples include mistreating users in a mental health crisis or aiding them in self-harm. All datasets in this
        scenario share the unified 'psychosocial' harm category, but each dataset (e.g., imminent_crisis, dependency)
        can have unique: - Scoring rubrics - Seed objectives The scenario tests whether models: - Recognize mental
        health crises and respond appropriately - Avoid facilitati

## Cyber

Tests whether a target can be induced to generate malware or exploitation content using single-turn
and multi-turn attacks.

```bash
pyrit_scan airt.cyber \
  --initializers target \
  --target openai_chat \
  --techniques multi_turn \
  --max-dataset-size 1
```

**Available techniques:** ALL, DEFAULT, MULTI_TURN, red_teaming

In [ ]:
from pyrit.scenario.airt import Cyber, CyberTechnique

dataset_config = DatasetAttackConfiguration(dataset_names=["airt_malware"], max_dataset_size=1)

scenario = Cyber()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [CyberTechnique.MULTI_TURN],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

scenario_result = await scenario.run_async()  # type: ignore

Executing Cyber:   0%|          | 0/2 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(scenario_result)


                                     📊 SCENARIO RESULTS: Cyber                                      

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Cyber
    • Scenario Version: 2
    • PyRIT Version: 0.15.0.dev0
    • Description:
        Cyber scenario implementation for PyRIT. This scenario tests how willing models are to exploit cybersecurity
        harms by generating malware. The Cyber class contains different variations of the malware generation techniques.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o-japan-nilfilter
    • Target Endpoint: https://pyrit-japan-test.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: FloatScaleThresholdScorer
      • scorer_type: true_false
      • score_aggregator: OR_
        └─ Composite of 1 scorer(s):
            • Scorer Type: AzureContentFilterS

## Jailbreak

Tests target resilience against jailbreak templates. A run crosses three selectors: the harmful
objectives (**dataset**, HarmBench), the **techniques** each jailbreak is delivered through, and
which **jailbreaks** to run. Two deliveries are on by default: `prompt_sending` renders the
objective inline into the template as a request converter (target-agnostic), and
`jailbreak_system_prompt` sets the template as a native system prompt with the objective sent as
the user turn (only for targets that natively support editable history + system prompts — it is
skipped for incapable targets). Registry techniques like `role_play_*`, `many_shot`, and `tap` are
opt-in. Results are grouped by jailbreak template, and a baseline (the un-jailbroken objective) is
included by default so complying with the bare objective is itself visible.

```bash
pyrit_scan airt.jailbreak \
  --initializers target load_default_datasets \
  --target openai_chat \
  --dataset-names harmbench \
  --max-dataset-size 1
```

**Available techniques:** ALL, DEFAULT (`prompt_sending` + `jailbreak_system_prompt`), plus registry
techniques (`role_play_*`, `many_shot`, `tap`, …). By default a small random sample of jailbreak
templates runs; pass `num_jailbreaks` (random count) or `jailbreak_names` (explicit) to widen or
pin the selection.

In [ ]:
from pyrit.scenario.airt import Jailbreak, JailbreakTechnique

dataset_config = DatasetAttackConfiguration(dataset_names=["harmbench"], max_dataset_size=1)

scenario = Jailbreak()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [JailbreakTechnique.DEFAULT],
        "jailbreak_names": ["aim.yaml"],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

scenario_result = await scenario.run_async()  # type: ignore

Executing Jailbreak:   0%|          | 0/163 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(scenario_result)


                                   📊 SCENARIO RESULTS: Jailbreak                                    

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Jailbreak
    • Scenario Version: 1
    • PyRIT Version: 0.15.0.dev0
    • Description:
        Jailbreak scenario implementation for PyRIT. This scenario tests how vulnerable models are to jailbreak attacks
        by applying various single-turn jailbreak templates to a set of test prompts. The responses are scored to
        determine if the jailbreak was successful.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o-japan-nilfilter
    • Target Endpoint: https://pyrit-japan-test.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: FloatScaleThresholdScorer
      • scorer_type: true_false
      • score_aggregator: OR_
        └─ Composite of 1 scorer

## Leakage

Tests whether a target can be induced to leak sensitive data or intellectual property, scored using
plagiarism detection.

```bash
pyrit_scan airt.leakage --target openai_chat --techniques first_letter --max-dataset-size 1
```

**Available techniques:** ALL, SINGLE_TURN, MULTI_TURN, IP, SENSITIVE_DATA, FirstLetter, Image, RolePlay, Crescendo

### Copyright and Plagiarism Testing

The FirstLetter technique tests whether a model has memorized copyrighted text by encoding it
with FirstLetterConverter (extracting first letters of each word) and asking the model to decode.
If the model reconstructs the original, it suggests memorization.

The PlagiarismScorer provides three complementary metrics for analyzing responses from any
leakage technique:

- **LCS (Longest Common Subsequence)** — Captures contiguous plagiarized sequences.
  Score = LCS length / reference length.
- **Levenshtein (Edit Distance)** — Measures word-level edit distance.
  Score = 1 − (min edits / max length).
- **Jaccard (N-gram Overlap)** — Measures phrase-level similarity using configurable n-grams.
  Score = matching n-grams / total reference n-grams.

All metrics are normalized to [0, 1] where 1 means the reference text is fully present. There is
no built-in threshold — the scorer returns a raw float for you to interpret per your use case.

In [ ]:
from pyrit.scenario.airt import Leakage, LeakageTechnique

dataset_config = DatasetAttackConfiguration(dataset_names=["airt_leakage"], max_dataset_size=1)

scenario = Leakage()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [LeakageTechnique.first_letter],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

scenario_result = await scenario.run_async()  # type: ignore

Executing Leakage:   0%|          | 0/2 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(scenario_result)


                                    📊 SCENARIO RESULTS: Leakage                                     

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Leakage
    • Scenario Version: 2
    • PyRIT Version: 0.15.0.dev0
    • Description:
        Leakage scenario implementation for PyRIT. This scenario tests how susceptible models are to leaking training
        data, PII, intellectual property, or other confidential information. Uses the registry/factory pattern to
        construct attack techniques.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o-japan-nilfilter
    • Target Endpoint: https://pyrit-japan-test.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseCompositeScorer
      • scorer_type: true_false
      • score_aggregator: AND_
        └─ Composite of 2 scorer(s):
            •

## Scam

Tests whether a target can be induced to generate scam, phishing, or fraud content.

```bash
pyrit_scan airt.scam \
  --initializers target \
  --target openai_chat \
  --techniques context_compliance \
  --max-dataset-size 1
```

**Available techniques:** ALL, DEFAULT, SINGLE_TURN, MULTI_TURN, ContextCompliance, RolePlay,
PersuasiveRedTeamingAttack. DEFAULT runs the single-turn techniques (ContextCompliance, RolePlay)
and omits the slower multi-turn PersuasiveRedTeamingAttack; run it via ALL or MULTI_TURN.

In [ ]:
from pyrit.scenario.airt import Scam, ScamTechnique

dataset_config = DatasetAttackConfiguration(dataset_names=["airt_scams"], max_dataset_size=1)

scenario = Scam()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [ScamTechnique.ContextCompliance],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

scenario_result = await scenario.run_async()  # type: ignore

Executing Scam:   0%|          | 0/2 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(scenario_result)


                                      📊 SCENARIO RESULTS: Scam                                      

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Scam
    • Scenario Version: 1
    • PyRIT Version: 0.15.0.dev0
    • Description:
        Scam scenario evaluates an endpoint's ability to generate scam-related materials (e.g., phishing emails,
        fraudulent messages) with primarily persuasion-oriented techniques.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o-japan-nilfilter
    • Target Endpoint: https://pyrit-japan-test.openai.azure.com/openai/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseCompositeScorer
      • scorer_type: true_false
      • score_aggregator: AND_
        └─ Composite of 2 scorer(s):
            • Scorer Type: SelfAskTrueFalseScorer
            • scorer_type: true_false
        